# MAGS — Online Hotel-Booking Platform (Polyglot Persistence)

**Module:** BEMM459J — Database Technologies for Business Analytics, University of Exeter Business School
**Type:** Group coursework (4 people). See the README for the full team and an honest breakdown of my contribution.

---

## Business problem
An online travel agency, *MAGS*, needs a back end that can do two different jobs well:

1. **Transactional bookings** — customers, reservations, rooms, invoices and feedback, where correctness and referential integrity matter (you cannot invoice a booking that does not exist).
2. **Fast catalogue search** — a flexible, read-heavy hotel directory (name, star rating, city, country) that changes shape over time and needs to be queried quickly.

A single database is a poor fit for both. This project demonstrates a **polyglot-persistence** design that uses the right store for each job.

## Solution & tech stack
| Layer | Technology | Why |
|---|---|---|
| Transactional core | **SQLite** (relational, 3NF) | ACID guarantees, foreign keys, joins for invoicing/analytics |
| Catalogue / search | **MongoDB** (document store) | schema-flexible, fast reads for a hotel directory |
| Application & analytics | **Python** — `sqlite3`, `pymongo`, `pandas`, `seaborn` | CRUD layer + descriptive analytics |

## My contribution (Sachin Sharma)
I owned the **relational layer and the analytics**: the 3NF schema design for the six core tables, the Python CRUD helpers over SQLite, the invoice-derivation logic (duration × nightly price), and the booking/ratings analysis at the end of this notebook. The MongoDB module was built collaboratively. *Published with my teammates' consent.*

## Results at a glance
- A normalised **6-table 3NF schema** (Customer, Booking, Room, Hotel, Invoice, Feedback) with enforced foreign keys.
- A reusable Python **CRUD layer** plus automated **invoice generation** (`Amount = nights × nightly price`).
- Descriptive analytics over synthetic data: **average bill ≈ £1.4k–£2.1k per stay**, **bookings-per-hotel** frequency, and a **hotel ranking by guest rating** (top hotels score 5/5).
- A parallel **MongoDB** document collection for the searchable hotel catalogue.

> **Reproducibility note.** This notebook runs top-to-bottom with no manual input. The relational database is rebuilt from [`sql/seed.sql`](sql/seed.sql) (run it once before this notebook, or use the setup cell below). The interactive admin menu now lives in [`app.py`](app.py). The MongoDB section falls back to an in-memory [`mongomock`](https://github.com/mongomock/mongomock) client when no live MongoDB server is present, so the notebook executes cleanly in CI. All data is **synthetic**.


## 1. Setup & database build

We import the analysis stack and (re)build `data/hotel.db` from the seed script so
the notebook is self-contained. `sqlite3` and `pandas` do the heavy lifting.


In [ ]:
import os
import sqlite3
import datetime

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# Locate the project root (this notebook lives in notebooks/).
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "sql")):
    PROJECT_ROOT = os.getcwd()  # fallback if run from the repo root

DB_PATH = os.path.join(PROJECT_ROOT, "data", "hotel.db")
SEED_SQL = os.path.join(PROJECT_ROOT, "sql", "seed.sql")
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

In [ ]:
# Rebuild the database from the seed script so this notebook is reproducible.
# (Equivalent to:  sqlite3 data/hotel.db < sql/seed.sql)
with sqlite3.connect(DB_PATH) as setup_conn, open(SEED_SQL) as fh:
    setup_conn.executescript(fh.read())

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

# A clean schema means no orphaned foreign keys:
orphans = conn.execute("PRAGMA foreign_key_check;").fetchall()
print("Foreign-key violations:", len(orphans))
print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")])

## 2. Relational CRUD layer

The helpers below wrap the common create / read / update / delete operations
against SQLite.

**Security note — parameterised queries.** All *values* are passed as `?`
placeholders and bound by the driver, which is the correct defence against SQL
injection. Table/column *names* (which cannot be bind parameters) are restricted
to a fixed allow-list. The original coursework used string concatenation here;
this version has been hardened. The same helpers are packaged in
[`db.py`](db.py) for reuse by the console app.


In [ ]:
# Allow-list of valid identifiers: table -> ordered columns.
SCHEMA = {
    "Customer": ["CustomerID", "First_name", "Last_name", "Address",
                 "Postal_code", "Contact_number", "Age"],
    "Booking":  ["BookingID", "Arrival_date", "Checkout_date", "Cancellation",
                 "Duration", "Number_of_guests", "Meal",
                 "CustomerCustomerID", "HotelHotelID"],
    "Hotel":    ["HotelID", "Name", "Contact", "Address",
                 "Postal_code", "Parking_space"],
    "Invoice":  ["InvoiceID", "BookingBookingID", "Amount",
                 "Discount", "Date", "Time"],
    "Room":     ["Room_number", "Room_type", "Price",
                 "HotelHotelID", "BookingBookingID"],
    "Feedback": ["FeedbackID", "BookingBookingID", "Feedback_text",
                 "Rating", "HotelHotelID"],
}

def _check(table, column=None):
    if table not in SCHEMA:
        raise ValueError(f"Unknown table: {table!r}")
    if column is not None and column not in SCHEMA[table]:
        raise ValueError(f"Unknown column {column!r} for {table!r}")

def insert_row(table, values):
    """Insert a full row using a parameterised query (values are bound, not concatenated)."""
    _check(table)
    cols = SCHEMA[table]
    placeholders = ", ".join("?" for _ in cols)
    sql = f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({placeholders})"
    conn.execute(sql, tuple(values))
    conn.commit()

def update_value(table, pk_col, pk_value, set_col, new_value):
    _check(table, pk_col); _check(table, set_col)
    conn.execute(f"UPDATE {table} SET {set_col} = ? WHERE {pk_col} = ?",
                 (new_value, pk_value))
    conn.commit()

def delete_row(table, pk_col, pk_value):
    _check(table, pk_col)
    conn.execute(f"DELETE FROM {table} WHERE {pk_col} = ?", (pk_value,))
    conn.commit()

def get_table(table):
    _check(table)
    return pd.read_sql_query(f"SELECT * FROM {table}", conn)

def get_row(table, pk_col, pk_value):
    _check(table, pk_col)
    return pd.read_sql_query(
        f"SELECT * FROM {table} WHERE {pk_col} = ?", conn, params=(pk_value,))

### CRUD in action

A quick round-trip to show the helpers work: read a table, insert a row, then
delete it again. (The full menu-driven admin tool lives in `app.py` so this
notebook never blocks on `input()`.)


In [ ]:
# Read: the Hotel catalogue as stored in the relational core.
get_table("Hotel")

In [ ]:
# Create then delete: add a feedback row, confirm the count, remove it.
before = len(get_table("Feedback"))
insert_row("Feedback", [111, 1000001, "Great stay, would book again.", 5, 4001])
after_insert = len(get_table("Feedback"))
delete_row("Feedback", "FeedbackID", 111)
after_delete = len(get_table("Feedback"))
print(f"Feedback rows -> before: {before}, after insert: {after_insert}, after delete: {after_delete}")

### Why parameterised queries matter

Below, a malicious string is supplied as a *value*. Because it is bound as a
parameter rather than concatenated into the SQL text, it is stored as harmless
literal data — the `Hotel` table is **not** dropped.


In [ ]:
malicious = "Smith'); DROP TABLE Hotel;--"
update_value("Customer", "CustomerID", 1001, "Last_name", malicious)
print("Hotel table still present with",
      len(get_table("Hotel")), "rows — injection neutralised.")
# Restore the original value.
update_value("Customer", "CustomerID", 1001, "Last_name", "Bottle")

## 3. Derived business logic — invoice generation

Two values are *derived* rather than entered by hand, mirroring how a real
billing process would work:

- **`Booking.Duration`** = nights between check-out and arrival.
- **`Invoice.Amount`** = `Duration × nightly Price` for the room on that booking.

The seed script already computes these in SQL; here we show the equivalent
logic and confirm the figures.


In [ ]:
# Recompute duration and invoice amount in SQL (idempotent).
conn.executescript("""
    UPDATE Booking
       SET Duration = CAST(julianday(Checkout_date) - julianday(Arrival_date) AS INTEGER);

    UPDATE Invoice
       SET Amount = (
            SELECT b.Duration * r.Price
              FROM Booking b
              JOIN Room    r ON r.BookingBookingID = b.BookingID
             WHERE b.BookingID = Invoice.BookingBookingID
       );
""")
conn.commit()

pd.read_sql_query("""
    SELECT i.InvoiceID, b.BookingID, b.Duration, r.Price, i.Amount, i.Discount
      FROM Invoice i
      JOIN Booking b ON b.BookingID = i.BookingBookingID
      JOIN Room    r ON r.BookingBookingID = b.BookingID
     ORDER BY i.InvoiceID
""", conn)

## 4. Analytics on the relational data

Three descriptive questions a MAGS analyst would ask of the transactional data.


### 4.1 What does a typical bill look like? (descriptive statistics)

In [ ]:
# Average bill, spread and discount across all bookings.
data_inv = pd.read_sql_query("SELECT * FROM Invoice", conn)
data_inv[["Amount", "Discount"]].describe()

### 4.2 How are bookings distributed across hotels?

In [ ]:
# Booking frequency per hotel.
data_book = pd.read_sql_query("SELECT * FROM Booking", conn)
(data_book.groupby("HotelHotelID")[["BookingID", "CustomerCustomerID"]]
          .count()
          .rename(columns={"BookingID": "num_bookings",
                           "CustomerCustomerID": "num_customers"}))

### 4.3 Which hotels are rated highest by guests?

In [ ]:
# Rank hotels by guest rating (highest first).
data_feed = pd.read_sql_query("SELECT * FROM Feedback", conn)
print("Top-ranked hotels by rating:")
data_feed[["HotelHotelID", "Rating", "Feedback_text"]].sort_values(
    by="Rating", ascending=False).reset_index(drop=True)

### 4.4 Ratings at a glance

In [ ]:
# Visualise each hotel's rating.
sns.set_style("whitegrid")
ax = sns.barplot(x="HotelHotelID", y="Rating", data=data_feed, color="orange")
ax.set_title("Guest rating by hotel")
ax.set_xlabel("Hotel ID")
ax.set_ylabel("Rating (1-5)")
plt.tight_layout()
plt.show()

## 5. NoSQL layer — MongoDB hotel catalogue

The searchable hotel directory is modelled as MongoDB documents. A document
store suits this read-heavy, schema-flexible catalogue, whereas the
transactional core above stays relational.

**Running without a database daemon.** The cell below tries to reach a local
MongoDB server; if none is running it transparently falls back to an in-memory
[`mongomock`](https://github.com/mongomock/mongomock) client, so the notebook
runs end-to-end in CI. In the original coursework this section used live
`pymongo` plus `ipywidgets` text boxes for data entry; the data-entry widgets
have been replaced with plain dictionary inserts so the notebook is
non-interactive.


In [ ]:
# Connect to MongoDB if available; otherwise use an in-memory mongomock client.
USING_MONGOMOCK = False
try:
    import pymongo
    client = pymongo.MongoClient("mongodb://localhost:27017/",
                                 serverSelectionTimeoutMS=800)
    client.server_info()  # forces a connection attempt
    print("Connected to a live MongoDB server.")
except Exception:
    import mongomock
    client = mongomock.MongoClient()
    USING_MONGOMOCK = True
    print("No live MongoDB found - using an in-memory mongomock client.")

hotel_db = client["hotel_db"]
hotel_records = hotel_db["hotel_records"]
hotel_records.delete_many({})  # start clean

In [ ]:
def save_hotel(record):
    """Insert one hotel document into the catalogue."""
    result = hotel_records.insert_one(record)
    return result.inserted_id is not None

# Seed the catalogue (replaces the interactive widget form).
catalogue = [
    {"ID": "4002", "NAME": "Dorsett City London",
     "STAR": "4", "CITY": "London", "COUNTRY": "UK"},
    {"ID": "4005", "NAME": "Grand Royale London Hyde Park",
     "STAR": "4", "CITY": "London", "COUNTRY": "UK"},
    {"ID": "4010", "NAME": "Hampton by Hilton London Waterloo",
     "STAR": "3", "CITY": "London", "COUNTRY": "UK"},
]
for hotel in catalogue:
    save_hotel(hotel)

print("Documents in catalogue:", hotel_records.count_documents({}))

In [ ]:
# Read back the catalogue as a DataFrame (a typical search/read operation).
results = hotel_records.find()
pd.DataFrame(results, columns=["ID", "NAME", "STAR", "CITY", "COUNTRY"])

In [ ]:
# Delete: documents can be removed by any field (here, by ID).
def delete_hotel(hotel_id):
    result = hotel_records.delete_one({"ID": hotel_id})
    return result.deleted_count

print("Deleted:", delete_hotel("4010"), "document(s)")
print("Remaining:", hotel_records.count_documents({}))

## 6. Summary & limitations

**What this notebook demonstrated**
- A normalised 6-table 3NF relational schema with enforced foreign keys.
- A parameterised Python CRUD layer over SQLite.
- Derived invoicing (`Amount = nights × nightly price`).
- Booking and ratings analytics.
- A parallel MongoDB document catalogue for searchable hotel data.

**Limitations & next steps** (see the README for the full list)
- **Synthetic data** only — figures are illustrative, not real bookings.
- **Single-node** SQLite/MongoDB — no replication, sharding or concurrency tuning.
- **No authentication / access control** — this is a coursework data layer, not a production service.
- Next: add automated tests, an API layer, and an end-to-end sync between the relational core and the search catalogue.


In [ ]:
conn.close()
print("Done.")